In [5]:
pip -q install -U autoawq transformers accelerate

In [6]:
import torch
from awq import AutoAWQForCausalLM
from transformers import AutoTokenizer, AutoConfig
import gc
import os

# Configure environment
os.environ["TOKENIZERS_PARALLELISM"] = "false"

base_model = "TinyLlama/TinyLlama-1.1B-Chat-v1.0"

print("Checking PyTorch and CUDA versions...")
print(f"PyTorch version: {torch.__version__}")
print(f"CUDA available: {torch.cuda.is_available()}")
if torch.cuda.is_available():
    print(f"CUDA version: {torch.version.cuda}")
    print(f"GPU: {torch.cuda.get_device_name()}")

# Clear any existing cache
if torch.cuda.is_available():
    torch.cuda.empty_cache()
gc.collect()

print("Loading tokenizer...")
tok = AutoTokenizer.from_pretrained(
    base_model,
    use_fast=True,
    trust_remote_code=True
)
if tok.pad_token is None:
    tok.pad_token = tok.eos_token

print("Loading model...")
# Load model with specific configurations to avoid attention issues
mdl = AutoAWQForCausalLM.from_pretrained(
    base_model,
    low_cpu_mem_usage=True,
    use_cache=False,
    trust_remote_code=True,
    torch_dtype=torch.float16,  # Use fp16 explicitly
    device_map={"": 0} if torch.cuda.is_available() else "cpu"  # More specific device mapping
)

# Set model to eval mode
mdl.eval()

# AWQ quantization config
quant_config = {
    "zero_point": True,
    "q_group_size": 128,
    "w_bit": 4,
    "version": "GEMM"
}

# Create more diverse and shorter calibration data
print("Preparing calibration data...")
calib_texts = [
    "The quick brown fox jumps over the lazy dog.",
    "Machine learning models process data efficiently.",
    "Natural language understanding is advancing rapidly.",
    "Deep neural networks learn complex patterns.",
    "Artificial intelligence transforms technology.",
    "Computer vision recognizes objects accurately.",
    "Robotics integrates sensors and actuators.",
    "Algorithm optimization improves performance significantly.",
    "Data science extracts meaningful insights.",
    "Software engineering creates reliable systems."
] * 10  # 100 samples total

print("Starting quantization...")
try:
    # Use minimal configuration to avoid batch size issues
    mdl.quantize(
        tok,
        quant_config=quant_config,
        calib_data=calib_texts,
        max_calib_seq_len=128,     # Reduced sequence length
        max_calib_samples=50,      # Reduced sample count
        n_parallel_calib_samples=1 # Keep sequential processing
    )

    print("Quantization completed successfully!")

    # Save the quantized model
    out_dir = "tinyllama-1.1b-awq"
    print(f"Saving model to {out_dir}...")

    mdl.save_quantized(out_dir, safetensors=True)
    tok.save_pretrained(out_dir)

    # Save config
    config = AutoConfig.from_pretrained(base_model, trust_remote_code=True)
    config.save_pretrained(out_dir)

    print(f"Model successfully quantized and saved to {out_dir}")

except Exception as e:
    print(f"Quantization failed with error: {str(e)}")
    print("\nTrying alternative approach with different model loading...")

    # Alternative approach: Load model differently
    del mdl
    if torch.cuda.is_available():
        torch.cuda.empty_cache()
    gc.collect()

    try:
        # Try loading without device_map first
        mdl = AutoAWQForCausalLM.from_pretrained(
            base_model,
            low_cpu_mem_usage=True,
            trust_remote_code=True,
            use_cache=False
        )

        # Move to device manually if CUDA is available
        if torch.cuda.is_available():
            mdl = mdl.cuda()

        mdl.eval()

        # Try with even smaller calibration parameters
        print("Attempting quantization with minimal parameters...")
        mdl.quantize(
            tok,
            quant_config=quant_config,
            calib_data=calib_texts[:20],  # Only use first 20 samples
            max_calib_seq_len=64,         # Even smaller sequence length
            max_calib_samples=20,         # Minimal samples
            n_parallel_calib_samples=1
        )

        out_dir = "tinyllama-1.1b-awq"
        mdl.save_quantized(out_dir, safetensors=True)
        tok.save_pretrained(out_dir)

        config = AutoConfig.from_pretrained(base_model, trust_remote_code=True)
        config.save_pretrained(out_dir)

        print(f"Model successfully quantized with alternative approach and saved to {out_dir}")

    except Exception as e2:
        print(f"Alternative approach also failed: {str(e2)}")
        print("\nConsider using a pre-quantized model instead:")
        print("TheBloke/TinyLlama-1.1B-Chat-v1.0-AWQ")

finally:
    # Cleanup
    if torch.cuda.is_available():
        torch.cuda.empty_cache()
    gc.collect()

Checking PyTorch and CUDA versions...
PyTorch version: 2.14.1+cu130
CUDA available: True
CUDA version: 13.0
GPU: Tesla T4
Loading tokenizer...
Loading model...


Reconstructing (incomplete total...): |          |  0.00B /  0.00B            

Fetching 10 files:   0%|          | 0/10 [00:00<?, ?it/s]

Loading weights:   0%|          | 0/201 [00:00<?, ?it/s]

Preparing calibration data...
Starting quantization...


AWQ: 100%|██████████| 22/22 [03:52<00:00, 10.55s/it]

Quantization completed successfully!
Saving model to tinyllama-1.1b-awq...


Writing model shards: 0it [00:00, ?it/s]

Model successfully quantized and saved to tinyllama-1.1b-awq


In [7]:
from awq import AutoAWQForCausalLM
from transformers import AutoTokenizer, TextStreamer
import torch

quantized_model_path = "tinyllama-1.1b-awq"

print("Loading quantized AWQ model...")
# Load the quantized model
model = AutoAWQForCausalLM.from_quantized(
    quantized_model_path,
    fuse_layers=True,
    trust_remote_code=True,
    device_map="auto"
)

# Load the corresponding tokenizer
tokenizer = AutoTokenizer.from_pretrained(quantized_model_path, trust_remote_code=True)

# Set up TinyLlama Chat Template prompt
prompt = "Explain the concept of quantum computing in one simple sentence."

chat = [
    {"role": "system", "content": "You are a helpful and concise AI assistant."},
    {"role": "user", "content": prompt}
]

# Apply the chat template to format the prompt correctly for TinyLlama
formatted_prompt = tokenizer.apply_chat_template(chat, tokenize=False, add_generation_prompt=True)

# Tokenize inputs and move them to GPU
inputs = tokenizer(formatted_prompt, return_tensors="pt").to("cuda")

# Streamer allows us to see the words printed as they are generated
streamer = TextStreamer(tokenizer, skip_prompt=True, skip_special_tokens=True)

print("\n--- Generating Response ---")
# Generate text
_ = model.generate(
    **inputs,
    max_new_tokens=256,
    temperature=0.7,
    top_p=0.9,
    do_sample=True,
    streamer=streamer
)

Loading quantized AWQ model...


Replacing layers...: 100%|██████████| 22/22 [00:11<00:00,  1.90it/s]
/usr/local/lib/python3.13/dist-packages/awq/models/base.py:541: UserWarning: Skipping fusing modules because AWQ extension is not installed.No module named 'awq_ext'
  warnings.warn("Skipping fusing modules because AWQ extension is not installed." + msg)



--- Generating Response ---
Quantum computing is the study and implementation of quantum systems, which operate beyond the classical limits of classical computing. This involves manipulating quantum bits (qubits) in ways that are beyond the reach of classical computers. Unlike classical computers, quantum computers can perform calculations much faster, but they also require specialized hardware and techniques to achieve this.


In [8]:
from awq import AutoAWQForCausalLM
from transformers import AutoTokenizer, TextStreamer
import torch

quantized_model_path = "tinyllama-1.1b-awq"

print("Loading quantized AWQ model...")
# Load the quantized model
model = AutoAWQForCausalLM.from_quantized(
    quantized_model_path,
    fuse_layers=True,
    trust_remote_code=True,
    device_map="auto"
)

# Load the corresponding tokenizer
tokenizer = AutoTokenizer.from_pretrained(quantized_model_path, trust_remote_code=True)

# Set up TinyLlama Chat Template prompt
prompt = "Explain the concept of quantum computing in one simple sentence."

chat = [
    {"role": "system", "content": "You are a helpful and concise AI assistant."},
    {"role": "user", "content": prompt}
]

# Apply the chat template to format the prompt correctly for TinyLlama
formatted_prompt = tokenizer.apply_chat_template(chat, tokenize=False, add_generation_prompt=True)

# Tokenize inputs and move them to GPU
inputs = tokenizer(formatted_prompt, return_tensors="pt").to("cuda")

# Streamer allows us to see the words printed as they are generated
streamer = TextStreamer(tokenizer, skip_prompt=True, skip_special_tokens=True)

print("\n--- Generating Response ---")
# Generate text
_ = model.generate(
    **inputs,
    max_new_tokens=256,
    temperature=0.7,
    top_p=0.9,
    do_sample=True,
    streamer=streamer
)

Loading quantized AWQ model...


Replacing layers...: 100%|██████████| 22/22 [00:12<00:00,  1.72it/s]



--- Generating Response ---
Quantum computing is a type of computing that operates using the principles of quantum mechanics, such as entanglement and superposition. It involves manipulating subatomic particles in ways that are difficult or impossible for classical computers to achieve. In quantum computing, calculations are performed in superposition, where multiple states are simultaneously available for a single quantum bit (qubit). This allows for more efficient and powerful computing than classical computers, and it could lead to breakthroughs in fields such as cryptography and drug discovery.
